In [2]:
import os
import pandas as pd
import numpy as np
from collections import Counter

BASE_DIR = ".."

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")

FILES = {
    "source1": os.path.join(TRAIN_DIR, "train_source1.tsv"),
    "source2": os.path.join(TRAIN_DIR, "train_source2.tsv"),
    "source3": os.path.join(TRAIN_DIR, "train_source3.tsv"),
    "ground_truth": os.path.join(TRAIN_DIR, "train_ground_truth.tsv"),
}

CHUNK_SIZE = 100_000

for name, path in FILES.items():
    print(f"{name:12} -> {path} -> {'FOUND' if os.path.exists(path) else 'MISSING'}")

source1      -> ../data/train/train_source1.tsv -> FOUND
source2      -> ../data/train/train_source2.tsv -> FOUND
source3      -> ../data/train/train_source3.tsv -> FOUND
ground_truth -> ../data/train/train_ground_truth.tsv -> FOUND


In [8]:
def profile_source_file(path, chunk_size=CHUNK_SIZE):
    total_rows = 0
    missing_name = 0
    missing_address = 0
    missing_country = 0
    
    country_counts = Counter()
    
    name_lengths = []
    address_lengths = []
    
    first_columns = None
    
    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False
    ):
        if first_columns is None:
            first_columns = list(chunk.columns)

        total_rows += len(chunk)
        
        missing_name += (chunk["business_name"].str.strip() == "").sum()
        missing_address += (chunk["business_address"].str.strip() == "").sum()
        missing_country += (chunk["country"].str.strip() == "").sum()
        
        country_counts.update(chunk["country"].str.strip())
        
        name_lengths.extend(chunk["business_name"].str.len().tolist())
        address_lengths.extend(chunk["business_address"].str.len().tolist())

    return {
        "rows": total_rows,
        "columns": first_columns,
        "missing_business_name": missing_name,
        "missing_business_address": missing_address,
        "missing_country": missing_country,
        "country_counts": country_counts,
        "name_lengths": name_lengths,
        "address_lengths": address_lengths,
    }


source_profiles = {}

for source in ["source1", "source2", "source3"]:
    print(f"Processing {source}...")
    source_profiles[source] = profile_source_file(FILES[source])
    print(f"Finished {source}")

Processing source1...
Finished source1
Processing source2...
Finished source2
Processing source3...
Finished source3


In [9]:
for source, profile in source_profiles.items():
    print("\n" + "=" * 60)
    print(source.upper())
    print("=" * 60)

    print("Rows:", profile["rows"])
    print("Columns:", profile["columns"])

    print("\nMissing values:")
    print("business_name :", profile["missing_business_name"])
    print("business_address:", profile["missing_business_address"])
    print("country       :", profile["missing_country"])

    print("\nCountries:")
    for country, count in profile["country_counts"].most_common():
        print(f"{country}: {count}")


SOURCE1
Rows: 2206821
Columns: ['entity_id', 'business_name', 'business_address', 'country']

Missing values:
business_name : 0
business_address: 0
country       : 0

Countries:
US: 1323633
India: 883188

SOURCE2
Rows: 5034616
Columns: ['entity_id', 'business_name', 'business_address', 'country']

Missing values:
business_name : 0
business_address: 168967
country       : 0

Countries:
US: 3016817
India: 2017799

SOURCE3
Rows: 5285603
Columns: ['entity_id', 'business_name', 'business_address', 'country']

Missing values:
business_name : 0
business_address: 175916
country       : 0

Countries:
US: 3170056
India: 2115547


In [10]:
def profile_text_fields(path, chunk_size=CHUNK_SIZE):
    total_rows = 0
    
    name_missing = 0
    address_missing = 0
    
    name_chars = 0
    address_chars = 0
    
    name_min = None
    name_max = None
    address_min = None
    address_max = None
    
    non_ascii_name = 0
    non_ascii_address = 0
    
    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False
    ):
        total_rows += len(chunk)

        names = chunk["business_name"]
        addresses = chunk["business_address"]

        name_lengths = names.str.len()
        address_lengths = addresses.str.len()

        name_missing += (names.str.strip() == "").sum()
        address_missing += (addresses.str.strip() == "").sum()

        name_chars += name_lengths.sum()
        address_chars += address_lengths.sum()

        current_name_min = name_lengths.min()
        current_name_max = name_lengths.max()

        current_address_min = address_lengths.min()
        current_address_max = address_lengths.max()

        if name_min is None or current_name_min < name_min:
            name_min = current_name_min

        if name_max is None or current_name_max > name_max:
            name_max = current_name_max

        if address_min is None or current_address_min < address_min:
            address_min = current_address_min

        if address_max is None or current_address_max > address_max:
            address_max = current_address_max

        non_ascii_name += names.map(
            lambda x: any(ord(c) > 127 for c in x)
        ).sum()

        non_ascii_address += addresses.map(
            lambda x: any(ord(c) > 127 for c in x)
        ).sum()

    non_empty_names = total_rows - name_missing
    non_empty_addresses = total_rows - address_missing

    return {
        "rows": total_rows,
        "name_missing": name_missing,
        "address_missing": address_missing,
        "name_avg_length": (
            name_chars / non_empty_names if non_empty_names else 0
        ),
        "address_avg_length": (
            address_chars / non_empty_addresses if non_empty_addresses else 0
        ),
        "name_min_length": name_min,
        "name_max_length": name_max,
        "address_min_length": address_min,
        "address_max_length": address_max,
        "non_ascii_name": non_ascii_name,
        "non_ascii_address": non_ascii_address,
    }

In [11]:
text_profiles = {}

for source in ["source1", "source2", "source3"]:
    print(f"Processing {source}...")
    text_profiles[source] = profile_text_fields(FILES[source])
    print(f"Finished {source}")

Processing source1...
Finished source1
Processing source2...
Finished source2
Processing source3...
Finished source3


In [12]:
pd.DataFrame(text_profiles).T

,rows,name_missing,address_missing,name_avg_length,address_avg_length,name_min_length,name_max_length,address_min_length,address_max_length,non_ascii_name,non_ascii_address
source1,2206821.0,0.0,0.0,24.034403,52.066213,3.0,105.0,11.0,256.0,0.0,554.0
source2,5034616.0,0.0,168967.0,25.103575,47.831173,2.0,104.0,0.0,249.0,764608.0,478453.0
source3,5285603.0,0.0,175916.0,25.202144,48.322719,2.0,123.0,0.0,240.0,606737.0,476588.0


In [13]:
def profile_ground_truth(path, chunk_size=CHUNK_SIZE):
    total_rows = 0
    
    total_s2_matches = 0
    total_s3_matches = 0
    total_matches = 0
    
    zero_matches = 0
    one_match = 0
    
    match_count_distribution = Counter()
    
    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False
    ):
        total_rows += len(chunk)

        for value in chunk["matched_entity_ids"]:
            value = value.strip()

            if value == "":
                match_count = 0
                zero_matches += 1
            else:
                ids = value.split(",")
                match_count = len(ids)

                for entity_id in ids:
                    if entity_id.startswith("S2-"):
                        total_s2_matches += 1
                    elif entity_id.startswith("S3-"):
                        total_s3_matches += 1

                total_matches += match_count

                if match_count == 1:
                    one_match += 1

            match_count_distribution[match_count] += 1

    return {
        "s1_entities": total_rows,
        "s2_matches": total_s2_matches,
        "s3_matches": total_s3_matches,
        "total_matches": total_matches,
        "zero_matches": zero_matches,
        "one_match": one_match,
        "distribution": match_count_distribution,
    }


gt_profile = profile_ground_truth(FILES["ground_truth"])

gt_profile

{'s1_entities': 2206821,
 's2_matches': 3693619,
 's3_matches': 3944746,
 'total_matches': 7638365,
 'zero_matches': 123247,
 'one_match': 119157,
 'distribution': Counter({3: 530841,
          4: 484115,
          2: 375212,
          5: 321957,
          6: 164868,
          0: 123247,
          1: 119157,
          7: 63968,
          8: 18680,
          9: 4205,
          10: 534,
          11: 37})}

In [14]:
distribution = pd.DataFrame(
    sorted(
        gt_profile["distribution"].items(),
        key=lambda x: x[0]
    ),
    columns=["match_count", "s1_entities"]
)

distribution["percentage"] = (
    distribution["s1_entities"]
    / gt_profile["s1_entities"]
    * 100
)

distribution

,match_count,s1_entities,percentage
0,0,123247,5.584821
1,1,119157,5.399486
2,2,375212,17.002376
3,3,530841,24.054556
4,4,484115,21.937212
5,5,321957,14.589176
6,6,164868,7.470837
7,7,63968,2.898649
8,8,18680,0.846466
9,9,4205,0.190546


In [15]:
print("Total Source 1 entities:", gt_profile["s1_entities"])
print("Total S2 matches:", gt_profile["s2_matches"])
print("Total S3 matches:", gt_profile["s3_matches"])
print("Total matches:", gt_profile["total_matches"])

print(
    "Singleton S1 entities:",
    gt_profile["zero_matches"],
    f"({gt_profile['zero_matches'] / gt_profile['s1_entities'] * 100:.2f}%)"
)

print(
    "S1 entities with exactly one match:",
    gt_profile["one_match"],
    f"({gt_profile['one_match'] / gt_profile['s1_entities'] * 100:.2f}%)"
)

Total Source 1 entities: 2206821
Total S2 matches: 3693619
Total S3 matches: 3944746
Total matches: 7638365
Singleton S1 entities: 123247 (5.58%)
S1 entities with exactly one match: 119157 (5.40%)


In [16]:
def source_match_distribution(path, chunk_size=CHUNK_SIZE):
    s1_with_s2 = 0
    s1_with_s3 = 0
    s1_with_both = 0
    s1_with_none = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False
    ):
        for value in chunk["matched_entity_ids"]:
            value = value.strip()

            if value == "":
                s1_with_none += 1
                continue

            ids = value.split(",")

            has_s2 = any(x.startswith("S2-") for x in ids)
            has_s3 = any(x.startswith("S3-") for x in ids)

            if has_s2:
                s1_with_s2 += 1

            if has_s3:
                s1_with_s3 += 1

            if has_s2 and has_s3:
                s1_with_both += 1

    return {
        "S1 with S2 match": s1_with_s2,
        "S1 with S3 match": s1_with_s3,
        "S1 with both": s1_with_both,
        "S1 with no match": s1_with_none,
    }


match_sources = source_match_distribution(FILES["ground_truth"])

pd.Series(match_sources)

S1 with S2 match    1919076
S1 with S3 match    1940545
S1 with both        1776047
S1 with no match     123247
dtype: int64

In [17]:
os.makedirs(os.path.join(BASE_DIR, "output"), exist_ok=True)

source_summary = []

for source in ["source1", "source2", "source3"]:
    p = source_profiles[source]
    t = text_profiles[source]

    source_summary.append({
        "source": source,
        "rows": p["rows"],
        "missing_business_name": p["missing_business_name"],
        "missing_business_address": p["missing_business_address"],
        "missing_country": p["missing_country"],
        "name_avg_length": t["name_avg_length"],
        "address_avg_length": t["address_avg_length"],
        "name_min_length": t["name_min_length"],
        "name_max_length": t["name_max_length"],
        "address_min_length": t["address_min_length"],
        "address_max_length": t["address_max_length"],
        "non_ascii_name": t["non_ascii_name"],
        "non_ascii_address": t["non_ascii_address"],
    })

source_summary_df = pd.DataFrame(source_summary)

source_summary_df.to_csv(
    os.path.join(BASE_DIR, "output", "eda_source_summary.csv"),
    index=False
)

distribution.to_csv(
    os.path.join(BASE_DIR, "output", "eda_match_distribution.csv"),
    index=False
)

print("EDA summaries saved.")

EDA summaries saved.
